In [1]:
from ultralytics import YOLO
import os
import torch
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
import random
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [2]:
def checking_gpu():
    print(torch.cuda.is_available())  # Should be True
    print(torch.cuda.get_device_name(0))  # Prints GPU name
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Using device: {device}")
    torch.cuda.is_available()

    return device

In [3]:
def model_training(model, device):
    results = model.train(
        project="YOLO11s-Experiments",
        name="left_right_signs_train",
        data="config.yaml",
        optimizer="AdamW",
        augment=True,
        fliplr=0.0,
        patience=7,
        epochs=100,  # Number of epochs
        imgsz=800,  # Image size
        batch=8,
        device=device,  # disable left-right flipsoo the left and right images wont get confused
        exist_ok=True,
        degrees=5,  # small tilt, like camera shake
        translate=0.1,  # slight shift
        scale=0.3,
        mosaic=0.0,
        mixup=0.0,
    )

    results = model.val()
    
    return results

In [4]:
def evaluation(results):

    # Print specific metrics
    print("Class indices with average precision:", results.ap_class_index)
    print("Average precision for all classes:", results.box.all_ap)
    print("Average precision:", results.box.ap)
    print("Average precision at IoU=0.50:", results.box.ap50)
    print("Class indices for average precision:", results.box.ap_class_index)
    print("Class-specific results:", results.box.class_result)
    print("F1 score:", results.box.f1)
    print("F1 score curve:", results.box.f1_curve)
    print("Overall fitness score:", results.box.fitness)
    print("Mean average precision:", results.box.map)
    print("Mean average precision at IoU=0.50:", results.box.map50)
    print("Mean average precision at IoU=0.75:", results.box.map75)
    print("Mean average precision for different IoU thresholds:", results.box.maps)
    print("Mean results for different metrics:", results.box.mean_results)
    print("Mean precision:", results.box.mp)
    print("Mean recall:", results.box.mr)
    print("Precision:", results.box.p)
    print("Precision curve:", results.box.p_curve)
    print("Precision values:", results.box.prec_values)
    print("Specific precision metrics:", results.box.px)
    print("Recall:", results.box.r)
    print("Recall curve:", results.box.r_curve)
   

In [5]:
def verify_yolo_labels_random(image_dir, label_dir, class_names, sample_limit=5):
    image_files = [f for f in os.listdir(image_dir) if f.endswith(('.jpg', '.png', '.jpeg'))]
    if len(image_files) == 0:
        print("No images found in the directory.")
        return

    # Select random sample
    random_images = random.sample(image_files, min(sample_limit, len(image_files)))

    for img_file in random_images:
        image_path = os.path.join(image_dir, img_file)
        label_file = os.path.splitext(img_file)[0] + ".txt"
        label_path = os.path.join(label_dir, label_file)

        if not os.path.exists(label_path):
            print(f"No label for {img_file}, skipping.")
            continue

        image = Image.open(image_path).convert("RGB")
        draw = ImageDraw.Draw(image)
        w, h = image.size

        with open(label_path, "r") as f:
            for line in f:
                cls, x, y, bw, bh = map(float, line.strip().split())
                cls = int(cls)
                x1 = (x - bw / 2) * w
                y1 = (y - bh / 2) * h
                x2 = (x + bw / 2) * w
                y2 = (y + bh / 2) * h
                draw.rectangle([x1, y1, x2, y2], outline="red", width=2)
                draw.text((x1, y1), class_names[cls], fill="white")

        plt.figure(figsize=(6, 6))
        plt.title(f"Labeled: {img_file}")
        plt.imshow(image)
        plt.axis("off")
        plt.show()



In [6]:
import torch
torch.cuda.empty_cache()

In [7]:
model = YOLO("yolov8m.pt")  
device=checking_gpu()

results = model_training(model,device)


In [ ]:
from huggingface_hub import login, upload_file
from dotenv import load_dotenv
import os
# Load .env file
load_dotenv()
# Log in with your HF token
login(token=os.getenv("HUGGINGFACE_TOKEN"))

# Define repo and local file
repo_id = "sue888888888888/yolo_road_signs_detection"
local_model_path = "runs/detect/road_signs_train24/weights/best.pt"

# Upload the model weights
upload_file(
    path_or_fileobj=local_model_path,
    path_in_repo="best.pt",
    repo_id=repo_id,
    repo_type="model"
)


best.pt:   0%|          | 0.00/6.24M [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/sue888888888888/yolo_road_signs_detection/commit/609dc5c1576674a09b11e5ab4713e76d3522cd86', commit_message='Upload best.pt with huggingface_hub', commit_description='', oid='609dc5c1576674a09b11e5ab4713e76d3522cd86', pr_url=None, repo_url=RepoUrl('https://huggingface.co/sue888888888888/yolo_road_signs_detection', endpoint='https://huggingface.co', repo_type='model', repo_id='sue888888888888/yolo_road_signs_detection'), pr_revision=None, pr_num=None)

In [10]:
from huggingface_hub import hf_hub_download
from ultralytics import YOLO

# Download the file
model_path = hf_hub_download(
    repo_id="sue888888888888/yolo_road_signs_detection",
    filename="best.pt",
    repo_type="model"
)

# Load with YOLO
model = YOLO(model_path)


best.pt:   0%|          | 0.00/6.24M [00:00<?, ?B/s]

In [ ]:
evaluation(model.val())